In [2]:
%load_ext autoreload
%autoreload 2

import json
import pathlib
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm.auto import tqdm
from src.dns_utils import resolve_asn, is_conclusive_asn

RAW_ASN = pathlib.Path("../data/interim/03_asn_raw.jsonl")
MAX_WORKERS = 10

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# --- 1) IPs únicos (o IP é do host NS; muitos domínios compartilham os mesmos)
ns_ips = pd.read_parquet("../data/interim/ns_resolver/ns_ips.parquet")
ips = ns_ips["ip"].dropna().unique()          # dropna: linhas de status não têm IP
print("IPs únicos:", len(ips))


IPs únicos: 18889


In [4]:
# --- 2) Retomada: pula IPs já resolvidos de forma conclusiva
done = set()
if RAW_ASN.exists():
    with RAW_ASN.open(encoding="utf-8") as f:
        done = {r["ip"] for r in map(json.loads, f) if is_conclusive_asn(r)}
todo = [ip for ip in ips if ip not in done]
print(f"já feitos: {len(done)} | a fazer: {len(todo)}")

já feitos: 18862 | a fazer: 27


In [5]:
# --- 3) Loop de enriquecimento
def safe_resolve_asn(ip):
    """Rede de segurança: bug inesperado vira registro de erro, não derruba o loop."""
    try:
        return resolve_asn(ip)
    except Exception as e:
        return {"ip": ip, "asn": None, "asn_all": None, "prefix": None,
                "cc": None, "registry": None, "as_name": None, "status": f"error: {e}"}

# Só a thread principal escreve no arquivo (evita corrida de escrita).
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool, RAW_ASN.open("a", encoding="utf-8") as f:
    futures = [pool.submit(safe_resolve_asn, ip) for ip in todo]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n")
        f.flush()

100%|██████████| 27/27 [00:01<00:00, 17.40it/s]


In [6]:
# --- 4) Bruto -> tabelas
asn_raw = pd.read_json(RAW_ASN, lines=True, dtype=False).drop_duplicates("ip", keep="last")
# dtype=False: não deixa o pandas "adivinhar" tipos; to_numeric + Int64 = inteiro que aceita nulo
asn_raw["asn"] = pd.to_numeric(asn_raw["asn"], errors="coerce").astype("Int64")

# 1 linha por IP
ip_asn = asn_raw[["ip", "asn", "asn_all", "prefix", "cc", "registry", "status"]]
# 1 linha por ASN (o nome é do ASN, não do IP)
asn_info = asn_raw.dropna(subset=["asn"])[["asn", "as_name"]].drop_duplicates("asn")

print(ip_asn["status"].value_counts())


# criar nova tabela de asn - orginfo detalhada (quebrando o as_name em handle, org, country_code (original segue padrão [handle  -  org  ,  c] ))
# PADRAO = r"^(?:(?P<handle>\S+) - )?(?P<org>.*?)(?:, (?P<country_code>[A-Z]{2}))?$"
PADRAO = r"^(?:(?P<handle>.+?) - )?(?P<org>.*?)(?:, (?P<country_code>[A-Z]{2}))?$"

asn_org_info = asn_info.copy()

partes = asn_org_info["as_name"].str.strip().str.extract(PADRAO)

# org vazia ("") não é informação: troca por NA para não confundir com org real
partes["org"] = partes["org"].replace("", pd.NA)

asn_org_info = asn_org_info.join(partes)   # join anexa as 3 colunas pelo índice
asn_org_info.to_parquet("../data/interim/asn_resolver/asn_org_info.parquet", index=False)

asn_org_info[["as_name", "handle", "org", "country_code"]].head()


# conferencia
asn_org_info[["handle", "org", "country_code"]].isna().sum()   # nulos por coluna
print("total de ASN:", len(asn_org_info))
print("total de ASN com país:", asn_org_info["country_code"].notna().sum())
print("total de ASN sem país:", asn_org_info["country_code"].isna().sum())
print("total de ASN com handle:", asn_org_info["handle"].notna().sum())
print("total de ASN sem handle:", asn_org_info["handle"].isna().sum())
print("total de ASN com org:", asn_org_info["org"].notna().sum())
print("total de ASN sem org:", asn_org_info["org"].isna().sum())


# inspect asn without org
display(asn_org_info[asn_org_info["org"].isna()].head(20))

display(asn_org_info[asn_org_info["handle"].isna()].head(20))

display(asn_org_info[asn_org_info["country_code"].isna()].head(20))

status
ok          18876
nxdomain       13
Name: count, dtype: int64
total de ASN: 1430
total de ASN com país: 1429
total de ASN sem país: 1
total de ASN com handle: 1429
total de ASN sem handle: 1
total de ASN com org: 1429
total de ASN sem org: 1


,asn,as_name,handle,org,country_code
774,54231,,NaN,NaN,NaN


,asn,as_name,handle,org,country_code
774,54231,,NaN,NaN,NaN


,asn,as_name,handle,org,country_code
774,54231,,NaN,NaN,NaN


In [7]:
ip_asn.to_parquet("../data/interim/asn_resolver/ip_asn.parquet", index=False)
asn_info.to_parquet("../data/interim/asn_resolver/asn_info.parquet", index=False)

In [8]:
# --- 5) Sanidade: domínios por ASN (encadeia domínio -> NS -> IP -> ASN)
ns_hosts = pd.read_parquet("../data/interim/ns_resolver/ns_hosts.parquet")
dom_asn = (ns_hosts[["domain", "ns"]]
           .merge(ns_ips[["ns", "ip"]], on="ns")      # NS -> IP
           .merge(ip_asn[["ip", "asn"]], on="ip"))    # IP -> ASN
(dom_asn.groupby("asn")["domain"].nunique()
        .sort_values(ascending=False).head(20)
        .rename("domains").reset_index()
        .merge(asn_info, on="asn"))

,asn,domains,as_name
0,13335,2297,"CLOUDFLARENET - Cloudflare, Inc., US"
1,16509,1446,"AMAZON-02 - Amazon.com, Inc., US"
2,44273,927,"GODADDY-DNS - Host Europe GmbH, DE"
3,8075,586,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...
4,12008,584,"VERCARA - Vercara, LLC, US"
5,21342,508,"AKAMAI-ASN2 - Akamai International B.V., NL"
6,16552,237,"DIGICERT - Tiggee LLC, US"
7,1921,216,"RCODEZERO-ANYCAST-SEC1-TLD - ipcom GmbH, AT"
8,2497,181,"IIJ - Internet Initiative Japan Inc., JP"
9,62597,169,"NSONE - NSONE Inc, US"


In [9]:
#check: numero de asn não resolvidos (nulos) e resolvidos (não nulos)
print("Total de ASN não resolvidos (nulos):", ip_asn["asn"].isna().sum())
print("Total de ASN resolvidos (não nulos):", ip_asn["asn"].notna().sum())

#check: numero de dominios unicos resolvidos (sem repetição)
print("Total de domínios resolvidos (sem repetição):", dom_asn["domain"].nunique())
print("Total de domínios não resolvidos (sem repetição):", ns_hosts["domain"].nunique() - dom_asn["domain"].nunique())

Total de ASN não resolvidos (nulos): 13
Total de ASN resolvidos (não nulos): 18876
Total de domínios resolvidos (sem repetição): 10224
Total de domínios não resolvidos (sem repetição): 9
